### Import modules and data

In [1]:
import pandas as pd
from pathlib import Path
import sys

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS
from backend.data.cleaning import clean_string_series

In [2]:
reservoirs_path = PATHS['raw_data_notebooks'] / 'reservoirs.csv'
reservoirs_pd = pd.read_csv(reservoirs_path)
reservoirs_pd.head()

,ID,SCOPE_NAME,RESERVOIR_NAME,TOTAL_WATER,ELECTRIC_FLAG
0,1,GUADALQUIVIR,"BREÑA, LA",103.0,0
1,3,GUADALQUIVIR,"FERNANDINA, LA",247.0,0
2,5,GUADALQUIVIR,PUEBLA DE CAZALLA,87.0,0
3,6,GUADALQUIVIR,PEDRO MARÍN,19.0,0
4,9,CUENCA MEDITERRÁNEA ANDALUZA,"VIÑUELA, LA",170.0,0


In [3]:
reservoirs_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 401 entries, 0 to 400
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              401 non-null    int64  
 1   SCOPE_NAME      401 non-null    object 
 2   RESERVOIR_NAME  401 non-null    object 
 3   TOTAL_WATER     400 non-null    float64
 4   ELECTRIC_FLAG   401 non-null    int64  
dtypes: float64(1), int64(2), object(2)
memory usage: 15.8+ KB


### Renaming columns

In [4]:
columns_dict = {'ID': 'id', 'SCOPE_NAME': 'scope', 'RESERVOIR_NAME': 'name', 'TOTAL_WATER':'capacity', 'ELECTRIC_FLAG': 'electric_flag'}
reservoirs_pd.rename(columns=columns_dict, inplace=True)

### Looking for Missing Values before cleaning texts and converting dtypes

In [5]:
reservoirs_pd.isna().sum()

id               0
scope            0
name             0
capacity         1
electric_flag    0
dtype: int64

### Converting capacity to Integer

In [6]:
# Let's ensure that every float is actually an int
print(f"Number of rows: {len(reservoirs_pd)}")
print(f"Number of int at capacity: {reservoirs_pd['capacity'].dropna().apply(float.is_integer).sum()}")

Number of rows: 401
Number of int at capacity: 400


In [7]:
reservoirs_pd.loc[reservoirs_pd['capacity'].notna(), 'capacity'] = reservoirs_pd.loc[reservoirs_pd['capacity'].notna(), 'capacity'].astype(int)

### Text Cleaning

In [8]:
reservoirs_pd.head()

,id,scope,name,capacity,electric_flag
0,1,GUADALQUIVIR,"BREÑA, LA",103.0,0
1,3,GUADALQUIVIR,"FERNANDINA, LA",247.0,0
2,5,GUADALQUIVIR,PUEBLA DE CAZALLA,87.0,0
3,6,GUADALQUIVIR,PEDRO MARÍN,19.0,0
4,9,CUENCA MEDITERRÁNEA ANDALUZA,"VIÑUELA, LA",170.0,0


In [9]:
reservoirs_pd['name'] = clean_string_series(reservoirs_pd['name'])
reservoirs_pd['scope'] = clean_string_series(reservoirs_pd['scope'])

### Save current cleaning and developing reservoir.ipynb at EDA folder

In [10]:
cleaned_reservoir_path = PATHS['pre_EDA'] / 'reservoir_for_EDA.csv'
cleaned_reservoir_path.parent.mkdir(parents=True, exist_ok=True)
reservoirs_pd.to_csv(cleaned_reservoir_path, index=False)


### Handling Missing Values

In [13]:
water_pd = pd.read_csv(PATHS['cleaned_data_notebooks'] / 'water_cleaned.csv')

# As it's the capacity of the reservoir, we will fill it with the maximum registered value
id_to_current_water = water_pd.groupby('id')['storage'].max() # Creates a series with 'ID' as index
mask = reservoirs_pd['capacity'].isna()
reservoirs_pd.loc[mask, 'capacity'] = reservoirs_pd.loc[mask, 'id'].map(id_to_current_water) # Replacing missing values efficiently

In [14]:
reservoirs_pd.head()

,id,scope,name,capacity,electric_flag
0,1,guadalquivir,brena,103.0,0
1,3,guadalquivir,fernandina,247.0,0
2,5,guadalquivir,puebla cazalla,87.0,0
3,6,guadalquivir,pedro marin,19.0,0
4,9,cuenca mediterranea andaluza,vinuela,170.0,0


In [15]:
reservoirs_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 401 entries, 0 to 400
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             401 non-null    int64  
 1   scope          401 non-null    object 
 2   name           401 non-null    object 
 3   capacity       401 non-null    float64
 4   electric_flag  401 non-null    int64  
dtypes: float64(1), int64(2), object(2)
memory usage: 15.8+ KB


### Saving cleaned data

In [16]:
cleaned_reservoirs_path = PATHS['cleaned_data_notebooks'] / 'reservoirs_cleaned.csv'
cleaned_reservoirs_path.parent.mkdir(parents=True, exist_ok=True)
reservoirs_pd.to_csv(cleaned_reservoirs_path, index=False)